# Figuras del informe TB1 · Grupo 4 (MovieLens 1M)

Este notebook genera con los datos reales de MovieLens 1M:

- **Figura 1.** Subgrafo usuario–película (aristas = calificaciones).
- **Figura 2.** Proyección usuario–usuario de esos mismos usuarios: dos usuarios se conectan si dieron 4 o 5 estrellas a por lo menos `s` películas en común (peso = películas en común).
- Las cifras del grafo completo (nodos, aristas usuario–película, aristas usuario–usuario y grupos con UFDS) para copiarlas al informe.
- (Opcional) un subgrafo por integrante, con usuarios de su propio rango.

Ejecuta todo con **Entorno de ejecución → Ejecutar todas**. Al final se descargan los PNG.

In [ ]:
# Celda 1: descargar MovieLens 1M (no se sube al repo por la licencia de GroupLens)
import os, urllib.request, zipfile
if not os.path.exists("ml-1m/ratings.dat"):
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-1m.zip", "ml-1m.zip")
    zipfile.ZipFile("ml-1m.zip").extractall(".")
print(os.listdir("ml-1m"))

In [ ]:
# Celda 2: parámetros
S = 20          # umbral de la arista usuario–usuario (igual que --s en main.py)
GUSTO = 4       # una calificación >= GUSTO cuenta como "le gustó"
N_PELICULAS = 12
USUARIOS_POR_INTEGRANTE = 4
MIN_PELICULAS_VISTAS = 6   # cada usuario de la figura calificó al menos 6 de las 12 películas
SEMILLA = 42
RANGOS = {"Rojas Gómez": (1, 2013), "Rivas Castillo": (2014, 4026), "Carhuaz Centeno": (4027, 6040)}

In [ ]:
# Celda 3: cargar los datos y construir el grafo bipartito completo (aristas usuario–película)
import pandas as pd
import networkx as nx

ratings = pd.read_csv("ml-1m/ratings.dat", sep="::", engine="python", encoding="latin-1",
                      names=["UserID", "MovieID", "Rating", "Timestamp"])
movies = pd.read_csv("ml-1m/movies.dat", sep="::", engine="python", encoding="latin-1",
                     names=["MovieID", "Title", "Genres"])
titulo = dict(zip(movies.MovieID, movies.Title))

G = nx.Graph()
G.add_nodes_from((f"U{u}" for u in ratings.UserID.unique()), tipo="usuario")
G.add_nodes_from((f"M{m}" for m in ratings.MovieID.unique()), tipo="pelicula")
G.add_weighted_edges_from(zip("U" + ratings.UserID.astype(str), "M" + ratings.MovieID.astype(str), ratings.Rating),
                          weight="rating")

n_usuarios = ratings.UserID.nunique()
n_peliculas = ratings.MovieID.nunique()
print("Usuarios:", n_usuarios)
print("Películas con al menos una calificación:", n_peliculas)
print("Nodos activos |V|:", G.number_of_nodes())
print("Aristas usuario–película (calificaciones):", G.number_of_edges())

In [ ]:
# Celda 4: aristas usuario–usuario en TODO el dataset (proyección del grafo bipartito)
# A[u, m] = 1 si el usuario u dio >= GUSTO a la película m. C = A·Aᵀ cuenta las películas
# que a ambos les gustaron; hay arista u–v si C[u, v] >= S.
import numpy as np

gustos = ratings[ratings.Rating >= GUSTO]
ids_u = np.sort(ratings.UserID.unique())
ids_m = np.sort(ratings.MovieID.unique())
fila = {u: i for i, u in enumerate(ids_u)}
col = {m: j for j, m in enumerate(ids_m)}

A = np.zeros((len(ids_u), len(ids_m)), dtype=np.float32)
A[gustos.UserID.map(fila).values, gustos.MovieID.map(col).values] = 1
C = A @ A.T
np.fill_diagonal(C, 0)

iu, iv = np.nonzero(np.triu(C >= S))
aristas_uu = list(zip(ids_u[iu], ids_u[iv], C[iu, iv].astype(int)))
print(f"Aristas usuario–usuario con s = {S}:", len(aristas_uu))
print("Usuarios con al menos una arista usuario–usuario:", len(set(ids_u[iu]) | set(ids_u[iv])))

In [ ]:
# Celda 5: UFDS (Union-Find) sobre las aristas usuario–usuario: cuántos grupos de gustos quedan
class UFDS:
    def __init__(self, elementos):
        self.padre = {x: x for x in elementos}
        self.rango = {x: 0 for x in elementos}
    def find(self, x):
        while self.padre[x] != x:
            self.padre[x] = self.padre[self.padre[x]]   # compresión de camino
            x = self.padre[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.rango[ra] < self.rango[rb]:
            ra, rb = rb, ra
        self.padre[rb] = ra
        if self.rango[ra] == self.rango[rb]:
            self.rango[ra] += 1
        return True

uf = UFDS(ids_u.tolist())
for u, v, _ in aristas_uu:
    uf.union(u, v)
from collections import Counter
grupos = Counter(uf.find(u) for u in ids_u)
tamanos = sorted(grupos.values(), reverse=True)
print(f"Grupos (componentes) con s = {S}:", len(grupos))
print("Grupo más grande:", tamanos[0], "usuarios")
print("Usuarios aislados (sin ninguna arista usuario–usuario):", sum(1 for t in tamanos if t == 1))

print("\nCómo cambia con el umbral s:")
for s in (10, 20, 30, 50):
    print(f"  s = {s:>2}: {int(np.triu(C >= s).sum()):>9} aristas usuario–usuario")

In [ ]:
# Celda 6: elegir los nodos de las figuras (criterio que se explica en el informe)
# Películas: las N_PELICULAS más calificadas. Usuarios: USUARIOS_POR_INTEGRANTE al azar (semilla fija)
# de cada rango, entre los que calificaron al menos MIN_PELICULAS_VISTAS de esas películas,
# para que ningún usuario quede casi sin aristas.
import random

top = ratings.MovieID.value_counts().head(N_PELICULAS).index.tolist()
vistas = ratings[ratings.MovieID.isin(top)].groupby("UserID").size()
candidatos = vistas[vistas >= MIN_PELICULAS_VISTAS].index

def elegir(desde, hasta, n, semilla=SEMILLA):
    pool = sorted(u for u in candidatos if desde <= u <= hasta)
    return sorted(random.Random(semilla).sample(pool, n))

elegidos = [u for (a, b) in RANGOS.values() for u in elegir(a, b, USUARIOS_POR_INTEGRANTE)]
integrante_de = {u: nombre for nombre, (a, b) in RANGOS.items() for u in elegidos if a <= u <= b}
print("Películas:", [titulo[m] for m in top])
print("Usuarios:", elegidos)

In [ ]:
# Celda 7: Figura 1, subgrafo usuario–película
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

COLORES = {1: "#d73027", 2: "#fc8d59", 3: "#bdbdbd", 4: "#91bfdb", 5: "#4575b4"}

def subgrafo(ids_usuarios, peliculas):
    sub = ratings[ratings.UserID.isin(ids_usuarios) & ratings.MovieID.isin(peliculas)]
    Sg = nx.Graph()
    Sg.add_nodes_from((f"U{u}" for u in ids_usuarios), tipo="usuario")
    Sg.add_nodes_from((f"M{m}" for m in peliculas), tipo="pelicula")
    for u, m, r in zip(sub.UserID, sub.MovieID, sub.Rating):
        Sg.add_edge(f"U{u}", f"M{m}", rating=r)
    return Sg

def dibujar(Sg, titulo_figura, archivo):
    us = [n for n, d in Sg.nodes(data=True) if d["tipo"] == "usuario"]
    ms = [n for n, d in Sg.nodes(data=True) if d["tipo"] == "pelicula"]
    pos = nx.bipartite_layout(Sg, us, align="vertical", scale=2)
    fig, ax = plt.subplots(figsize=(12, 8), dpi=200)
    for r, c in COLORES.items():
        aristas = [(a, b) for a, b, d in Sg.edges(data=True) if d["rating"] == r]
        nx.draw_networkx_edges(Sg, pos, edgelist=aristas, edge_color=c, width=1.3, alpha=0.85, ax=ax)
    nx.draw_networkx_nodes(Sg, pos, nodelist=us, node_color="#2f6fb0", node_size=450, ax=ax)
    nx.draw_networkx_nodes(Sg, pos, nodelist=ms, node_color="#e08a2c", node_shape="s", node_size=450, ax=ax)
    nx.draw_networkx_labels(Sg, pos, {u: u for u in us}, font_size=6, font_color="white", ax=ax)
    for m in ms:
        x, y = pos[m]
        ax.text(x + 0.08, y, titulo[int(m[1:])][:38], fontsize=7, va="center")
    leyenda = [Line2D([], [], marker="o", ls="", color="#2f6fb0", label="Usuario"),
               Line2D([], [], marker="s", ls="", color="#e08a2c", label="Película")]
    leyenda += [Line2D([], [], color=c, lw=2, label=f"Calificación {r}") for r, c in COLORES.items()]
    ax.legend(handles=leyenda, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=7, fontsize=8, frameon=False)
    ax.set_title(f"{titulo_figura}\n{len(us)} usuarios, {len(ms)} películas, {Sg.number_of_edges()} calificaciones",
                 fontsize=11)
    ax.axis("off")
    ax.set_xlim(ax.get_xlim()[0], ax.get_xlim()[1] + 1.2)
    fig.tight_layout()
    fig.savefig(archivo, bbox_inches="tight")
    plt.show()
    print("Guardada:", archivo)

S1 = subgrafo(elegidos, top)
dibujar(S1, "Figura 1. Subgrafo usuario–película (MovieLens 1M)", "figura1_usuario_pelicula.png")
grados = dict(S1.degree(f"U{u}" for u in elegidos))
print("Calificaciones de cada usuario en la figura:", grados)

In [ ]:
# Celda 8: Figura 2, aristas usuario–usuario entre los mismos 12 usuarios
# El peso sale de TODO el dataset (no solo de las 12 películas de la Figura 1).
COLOR_INTEGRANTE = dict(zip(RANGOS, ["#2f6fb0", "#3a9e5c", "#8e5bb5"]))

P = nx.Graph()
P.add_nodes_from(elegidos)
for i, u in enumerate(elegidos):
    for v in elegidos[i + 1:]:
        w = int(C[fila[u], fila[v]])
        if w >= S:
            P.add_edge(u, v, weight=w)

pos = nx.circular_layout(P)
fig, ax = plt.subplots(figsize=(9, 9), dpi=200)
pesos = [d["weight"] for _, _, d in P.edges(data=True)]
maximo = max(pesos, default=1)
nx.draw_networkx_edges(P, pos, width=[0.6 + 4.4 * w / maximo for w in pesos],
                       edge_color="#7a7a7a", alpha=0.7, ax=ax)
nx.draw_networkx_edge_labels(P, pos, {(a, b): d["weight"] for a, b, d in P.edges(data=True)},
                             font_size=6, label_pos=0.5, ax=ax)
nx.draw_networkx_nodes(P, pos, node_color=[COLOR_INTEGRANTE[integrante_de[u]] for u in P.nodes],
                       node_size=900, ax=ax)
nx.draw_networkx_labels(P, pos, {u: f"U{u}" for u in P.nodes}, font_size=7, font_color="white", ax=ax)
leyenda = [Line2D([], [], marker="o", ls="", color=c, markersize=10, label=f"Usuarios {a}–{b}")
           for (nombre, c), (a, b) in zip(COLOR_INTEGRANTE.items(), RANGOS.values())]
leyenda.append(Line2D([], [], color="#7a7a7a", lw=2, label=f"Arista: ≥ {S} películas con {GUSTO}–5 estrellas en común"))
ax.legend(handles=leyenda, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2, fontsize=8, frameon=False)
ax.set_title(f"Figura 2. Aristas usuario–usuario (s = {S})\n"
             f"{P.number_of_nodes()} usuarios, {P.number_of_edges()} aristas; el número es el peso", fontsize=11)
ax.axis("off")
fig.tight_layout()
fig.savefig("figura2_usuario_usuario.png", bbox_inches="tight")
plt.show()
print("Guardada: figura2_usuario_usuario.png")
print(f"Densidad: {nx.density(P):.2f} (1.0 = todos conectados con todos)")

In [ ]:
# Celda 9 (opcional): un subgrafo usuario–película por integrante, con usuarios de su rango
for i, (nombre, (a, b)) in enumerate(RANGOS.items(), start=1):
    Sg = subgrafo(elegir(a, b, 8), top[:10])
    dibujar(Sg, f"Subgrafo de {nombre} (usuarios {a} a {b})", f"subgrafo_integrante{i}.png")

In [ ]:
# Celda 10: cifras para el informe y descarga de las imágenes
print(f"Nodos activos: {G.number_of_nodes():,} ({n_usuarios:,} usuarios y {n_peliculas:,} películas)")
print(f"Aristas usuario–película: {G.number_of_edges():,}")
print(f"Aristas usuario–usuario (s = {S}): {len(aristas_uu):,}")
print(f"Grupos con UFDS (s = {S}): {len(grupos):,}; el mayor tiene {tamanos[0]:,} usuarios")
print(f"Figura 1: {len(elegidos)} usuarios ({USUARIOS_POR_INTEGRANTE} por integrante), {len(top)} películas, "
      f"{S1.number_of_edges()} calificaciones")
print(f"Figura 2: {P.number_of_edges()} aristas usuario–usuario entre esos {len(elegidos)} usuarios")

try:
    from google.colab import files
    for f in ["figura1_usuario_pelicula.png", "figura2_usuario_usuario.png",
              "subgrafo_integrante1.png", "subgrafo_integrante2.png", "subgrafo_integrante3.png"]:
        if os.path.exists(f):
            files.download(f)
except ImportError:
    print("Fuera de Colab: los PNG quedaron en la carpeta actual.")

## Texto sugerido para el informe

Reemplaza los valores entre corchetes con lo que imprime la Celda 10.

**2.3 Relación con grafos (párrafo nuevo, después del grafo bipartito).**
Además de las aristas usuario–película, se agrega un segundo tipo de conexión entre usuarios, obtenido como una proyección del grafo bipartito (Zhou et al., 2007). Dos usuarios quedan conectados cuando ambos asignaron 4 o 5 estrellas a por lo menos s = 20 películas en común, y el peso de la arista es la cantidad de esas películas. Estas aristas no se inventan: salen de las mismas calificaciones del dataset y representan afinidad de gustos. Con este criterio se obtienen [N] aristas usuario–usuario, que serán la base para agrupar usuarios con gustos similares mediante UFDS, uno de los objetivos del caso de estudio.

**3. Visualización (antes de la Figura 1).**
Para la figura se tomaron las 12 películas con más calificaciones del dataset y, de cada rango de usuarios asignado a un integrante, 4 usuarios elegidos al azar (semilla 42) entre los que calificaron al menos 6 de esas películas. Así todos los usuarios del subgrafo tienen conexiones suficientes para observar la estructura.

**3. Visualización (Figura 2).**
La Figura 2 muestra las aristas usuario–usuario entre los mismos 12 usuarios de la Figura 1. El grosor y el número de cada arista indican cuántas películas calificaron ambos con 4 o 5 estrellas en todo el dataset.

**Referencia.**
Zhou, T., Ren, J., Medo, M., & Zhang, Y.-C. (2007). Bipartite network projection and personal recommendation. *Physical Review E, 76*(4), 046115. https://doi.org/10.1103/PhysRevE.76.046115